In [1]:
!pip install langgraph langchain openai

In [2]:
!pip install langgraph langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
    Uninstalling langchain-core-1.2.17:
      Successfully uninstalled langchain-core-1.2.17


In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

In [4]:
# Agent state
class AgentState(TypedDict):
    email_content: str
    approved: bool

In [5]:
# 1️⃣ AI mail yazıyor
def write_email(state):
    email = "Merhaba, yarın toplantımız saat 10:00'da."
    print("AI mail oluşturdu:")
    print(email)

    return {"email_content": email}

In [6]:
# 2️⃣ Human-in-the-loop
def human_review(state):

    print("\nMail içeriği:")
    print(state["email_content"])

    decision = input("\nMail gönderilsin mi? (yes/no): ")

    if decision == "yes":
        return {"approved": True}
    else:
        return {"approved": False}

In [7]:
# 3️⃣ Mail gönderme
def send_email(state):

    print("\n📨 Mail gönderildi!")
    print(state["email_content"])

    return state

In [8]:
# Graph oluştur
builder = StateGraph(AgentState)

builder.add_node("write_email", write_email)
builder.add_node("human_review", human_review)
builder.add_node("send_email", send_email)

builder.set_entry_point("write_email")

builder.add_edge("write_email", "human_review")

In [9]:
# Koşullu geçiş
def decision(state):
    if state["approved"]:
        return "send_email"
    else:
        return END

In [10]:
builder.add_conditional_edges(
    "human_review",
    decision,
    {
        "send_email": "send_email",
        END: END
    }
)

In [11]:
builder.add_edge("send_email", END)

graph = builder.compile()

# Çalıştır
graph.invoke({})

AI mail oluşturdu:
Merhaba, yarın toplantımız saat 10:00'da.

Mail içeriği:
Merhaba, yarın toplantımız saat 10:00'da.

Mail gönderilsin mi? (yes/no): yes

📨 Mail gönderildi!
Merhaba, yarın toplantımız saat 10:00'da.


{'email_content': "Merhaba, yarın toplantımız saat 10:00'da.",
 'approved': True}